vw_dataset
- Plots
- Correlation Matrices
- Insights
- Feature-engineering Decisions

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from scipy.stats import shapiro, normaltest, anderson

In [ ]:
vws_dataset = pd.read_parquet(
    "../data/processed/vws_dataset.parquet"
)

In [ ]:
wf_mean_ds = pd.read_parquet(
    "../data/processed/wf_mean_ds.parquet"
)

In [ ]:
mean_weather_feature = pd.read_parquet(
    "../data/processed/mean_weather_feature.parquet"
)

<h1> for three frameworks

In [ ]:
vars_to_check = [
    "ws100",
    "wind_shear",
    "air_density",
    "relative_humidity"
]

for name, df in {
    "UK Mean" : mean_weather_feature,
    "Wind-Farm Mean" : wf_mean_ds,
    "GVWS": vws_dataset
}.items():

    print("\n", "="*50)
    print(name)
    print("="*50)

    print(
        df[vars_to_check]
        .describe()
        .loc[["mean","std","min","max"]]
    )

In [ ]:
sample = slice(0, 2000)

plt.figure(figsize=(14,6))

plt.plot(
    mean_weather_feature["ws100"].iloc[sample],
    label="UK Mean"
)

plt.plot(
    wf_mean_ds["ws100"].iloc[sample],
    label="Wind-Farm Mean"
)

plt.plot(
    vws_dataset["ws100"].iloc[sample],
    label="GVWS"
)

plt.legend()
plt.title("Comparison of Weather Representations (WS100)")
plt.show()

In [ ]:
#correaltion between wind speed at 100m and wind generation i.e. WIND
corr_mean = mean_weather_feature["ws100"].corr(
    mean_weather_feature["WIND"]
)

corr_wf = wf_mean_ds["ws100"].corr(
    wf_mean_ds["WIND"]
)

corr_gvws = vws_dataset["ws100"].corr(
    vws_dataset["WIND"]
)

print(corr_mean)
print(corr_wf)
print(corr_gvws)

In [ ]:
#for high-wind/storm events comparison
top_events = (
    vws_dataset
    .nlargest(500, "WIND")
)

top_events[["timestamp", "WIND"]].head()

In [ ]:
#taking one representative week
target_date = "2025-12-05"

start = pd.Timestamp(target_date) - pd.Timedelta(days=3)
end = pd.Timestamp(target_date) + pd.Timedelta(days=3)

In [ ]:
#wind generation and weather representation at high wind
fig, ax1 = plt.subplots(figsize=(14,6))

ax1.plot(
    gvws_plot["timestamp"],
    gvws_plot["WIND"],
    color="black",
    linewidth=3,
    label="Generation"
)

ax1.set_ylabel("Wind Generation (MW)")

ax2 = ax1.twinx()

ax2.plot(
    uk_plot["timestamp"],
    uk_plot["ws100"],
    label="UK Mean",
    alpha=0.8
)

ax2.plot(
    wf_plot["timestamp"],
    wf_plot["ws100"],
    label="Wind-Farm Mean",
    alpha=0.8
)

ax2.plot(
    gvws_plot["timestamp"],
    gvws_plot["ws100"],
    label="GVWS",
    alpha=0.8
)

ax2.set_ylabel("WS100 (m/s)")

fig.legend(loc="upper left")
plt.title("Wind Generation vs Weather Representations")

plt.show()

In [ ]:
#kde plot distribution comparision
plt.figure(figsize=(12,6))

sns.kdeplot(
    mean_weather_feature["ws100"],
    label="UK Mean",
    fill=True
)

sns.kdeplot(
    wf_mean_ds["ws100"],
    label="Wind-Farm Mean",
    fill=True
)

sns.kdeplot(
    vws_dataset["ws100"],
    label="GVWS",
    fill=True
)

plt.xlabel("WS100 (m/s)")
plt.title("Distribution Comparison of Weather Representations")
plt.legend()

plt.show()

In [ ]:
#comparing summary statistics
comparison = pd.DataFrame({
    "UK Mean": mean_weather_feature["ws100"].describe(),
    "Wind-Farm Mean": wf_mean_ds["ws100"].describe(),
    "GVWS": vws_dataset["ws100"].describe()
})

print(comparison.loc[
    ["mean","std","25%","50%","75%","max"]
])

<h1> VWS

In [ ]:
vws_dataset.head()

In [ ]:
vws_dataset.info()

In [ ]:
vws_dataset.isna().sum()

In [ ]:
vws_dataset.describe()

EDA of Target Variable

In [ ]:
vws_dataset["WIND"].describe()

In [ ]:
#histogram
plt.figure(figsize=(10,6))

sns.histplot(
    vws_dataset["WIND"],
    bins=40,
    kde=True,
    color="steelblue"
)

plt.title("Distribution of Wind Generation")
plt.xlabel("Wind Generation (MW)")
plt.ylabel("Frequency")

plt.show()

In [ ]:
#qq_plot

plt.figure(figsize=(8,6))

stats.probplot(vws_dataset["WIND"], dist="norm", plot=plt)

plt.title("Q-Q plot of Wind Generation")

plt.show()

In [ ]:
#skewness and kurtosis

skewness = stats.skew(vws_dataset["WIND"])
kurtosis = stats.kurtosis(vws_dataset["WIND"])

print(f"Skewness: {skewness: .3f}")
print(f"Kurtosis: {kurtosis: .3f}")

In [ ]:
f_sample = vws_dataset["WIND"]

stat, p = shapiro(f_sample)

print("Shapiro Wilk Test")
print(f"Statistics: {stat: .3f}")
print(f"P-value: {p: .4e}")

if p > 0.05:
    print("Fail to reject NH, it is normally distributed")
else:
    print("Reject NH, it is not distributed normally")

In [ ]:
#shapiro wilk test has best result for smaller dataset

t_sample = vws_dataset["WIND"].sample(5000, random_state=42)

stat, p = shapiro(t_sample)

print("Shapiro Wilk Test")
print(f"Statistics: {stat: .3f}")
print(f"P-value: {p: .4e}")

if p > 0.05:
    print("Fail to reject Null Hypothesis: It is normally distributed")
else:
    print("Reject Null Hypothese: It is not distributed normally")

In [ ]:
stat, p=normaltest(f_sample)

print("D'Angostino Normality Test")
print(f"Statistics: {stat: .4e}")
print(f"P-value: {p: .4e}")

if p > 0.05:
    print("Fail to reject null hypothesis, it is normally distributed")
else:
    print("Reject null hypothesis, it is not distributed normally")

In [ ]:
#anderson-darling test

result = anderson(f_sample, dist="norm")

print("Anderson-Darling Test")
print(f"Statistics: {result.statistic: .4f}")

for sl, cv in zip(result.significance_level, result.critical_values):
    print(f"Significance Level: {sl}& | Critical Value: {cv}")

if result.statistic.all() > result.critical_values.all():
    print("Reject normality assumption")
else:
    print("Appears Normally distributed")

In [ ]:
plt.figure(figsize=(10,2))

sns.boxplot(
    x=f_sample,
    color="skyblue"
)

plt.title("Boxplot of Wind Generation")

plt.show()

In [ ]:
plt.figure(figsize=(10,8))

sns.lineplot(
    vws_dataset,
    x="timestamp",
    y="WIND"    
)

plt.title("Wind generation (2018-2025)")
plt.xlabel("Timeframe")
plt.ylabel("Wind Generation (MW)")
plt.tight_layout()

plt.show()

In [ ]:
sns.histplot(vws_dataset["WIND"], bins=30)

In [ ]:
vws_dataset["hour"] = vws_dataset["timestamp"].dt.hour
sns.boxplot(vws_dataset, x = "hour", y= "WIND")

In [ ]:
vws_dataset["month"] = vws_dataset["timestamp"].dt.month

sns.boxplot(vws_dataset, x="month", y="WIND")

In [ ]:
monthly_avg = (
    vws_dataset
    .groupby(vws_dataset["timestamp"].dt.year)["WIND"]
    .mean()
)

plt.figure(figsize=(12,8))
monthly_avg.plot(kind="bar", color="maroon")
plt.title("Average Wind Generation Yearly")
plt.xlabel("Year")
plt.ylabel("Average Wind Generation")
plt.show()

In [ ]:
plt.figure(figsize=(14,6))

sns.regplot(
    vws_dataset,
    x=vws_dataset.index,
    y="WIND",
    scatter_kws={'s':5, 'alpha':0.2},
    line_kws={'color': 'red'}
)

Weather Variables

In [ ]:
#histogram

plt.figure(figsize=(10,6))


plt.subplot(1,2,1)
sns.histplot(
    vws_dataset["ws10"],
    bins=40
)
plt.xlabel("Wind Speed at 10m")
plt.title("Wind Speed at 10m")

plt.subplot(1,2,2)
sns.scatterplot(
    vws_dataset.sample(4000, random_state=42),
    x="ws10",
    y="WIND"
)
plt.xlabel("Wind Speed at 10m")
plt.ylabel("Wind Generation")
plt.title("Scatter plot of wind speed at 10m")

plt.tight_layout()
plt.show()

In [ ]:
#histogram

plt.figure(figsize=(10,6))


plt.subplot(1,2,1)
sns.histplot(
    vws_dataset["ws100"],
    bins=40
)
plt.xlabel("Wind Speed at 100m")
plt.title("Wind Speed at 100m")

plt.subplot(1,2,2)
sns.scatterplot(
    vws_dataset.sample(4000, random_state=42),
    x="ws100",
    y="WIND"
)
plt.xlabel("Wind Speed at 100m")
plt.ylabel("Wind Generation")
plt.title("Scatter plot of wind speed at 100m")

plt.tight_layout()
plt.show()

In [ ]:
vws_dataset["year"] = vws_dataset["timestamp"].dt.year

In [ ]:
plt.figure(figsize=(15,6))

plt.subplot(1,3,1)
sns.boxplot(
    vws_dataset,
    x="hour",
    y="ws100"
)
plt.xlabel("Hour")
plt.ylabel("Speed")

plt.subplot(1,3,2)
sns.boxplot(
    vws_dataset,
    x="month",
    y="ws100"
)
plt.xlabel("Month")
plt.ylabel(None)

plt.subplot(1,3,3)
sns.boxplot(
    vws_dataset,
    x="year",
    y="ws100"
)
plt.xlabel("Year")
plt.ylabel(None)

plt.suptitle("Boxplot of Wind Speed")
plt.tight_layout()

plt.show()

In [ ]:
#!pip install windrose

In [ ]:
from windrose import WindroseAxes

fig, ax= plt.subplots(
    1,2,
    figsize=(16,8),
    subplot_kw={"projection":"windrose"},
    squeeze=True
)

ax[0].bar(
    vws_dataset["wd100"],
    vws_dataset["ws100"],
    normed=True,
    opening=0.8,
    edgecolor="white"
)

ax[0].set_title("Wind direction at 100m", pad=25)
ax[0].legend(
    title="Wind Speed (m/s)",
    loc= "lower left",
    bbox_to_anchor = (-0.10, 0.0),
    handlelength = 0.8
)

ax[1].bar(
    vws_dataset["wd10"],
    vws_dataset["ws10"],
    normed=True,
    opening=0.8,
    edgecolor="white"
)

ax[1].set_title("Wind Direction at 10m", pad=25)
ax[1].legend(
    title="Wind Speed (m/s)",
    loc= "lower left",
    bbox_to_anchor = (-0.10, 0.0),
    handlelength = 0.8
)


plt.show()

In [ ]:
plt.figure(figsize=(12,4))

plt.subplot(1,2,1)
sns.histplot(
    vws_dataset["wind_shear"],
    bins=20
)

plt.xlabel("Wind Shear")
plt.ylabel("Count")
plt.title("Histogram of wind shear")

plt.subplot(1,2,2)
sns.scatterplot(
    vws_dataset.sample(2000, random_state=42),
    x="wind_shear",
    y="WIND"
)

plt.show()

In [ ]:
sort = vws_dataset["wind_shear"].sort_values()

In [ ]:
sort.tail(10)

In [ ]:
#air_density
plt.figure(figsize=(12,6))

plt.subplot(1,2,1)
sns.histplot(
    vws_dataset["air_density"],
    bins=40
)
plt.xlabel("Air Density")

plt.subplot(1,2,2)
sns.boxplot(
    vws_dataset,
    x="month",
    y="air_density"
)
plt.ylabel("Air Density")
plt.xlabel("Months")

plt.suptitle("Box plot of Air Density and its trend")
plt.tight_layout()
plt.show()

In [ ]:
#relative_humidity
plt.figure(figsize=(12,6))

plt.subplot(1,2,1)
sns.histplot(
    vws_dataset["relative_humidity"],
    bins=40
)
plt.xlabel("Relative Humidity")

plt.subplot(1,2,2)
sns.boxplot(
    vws_dataset,
    x="month",
    y="relative_humidity"
)
plt.ylabel("Relative Humidity")
plt.xlabel("Months")

plt.suptitle("Box plot of Relative Humidty and its trend")
plt.tight_layout()
plt.show()

In [ ]:
#blh

plt.figure(figsize=(18,6))

plt.subplot(1,3,1)
sns.histplot(
    vws_dataset["blh"],
    bins=40
)
plt.xlabel("Boundary Layer Height")

plt.subplot(1,3,2)
sns.boxplot(
    vws_dataset,
    x="month",
    y="blh"
)
plt.ylabel("Boundary Layer Height")
plt.xlabel("Months")

plt.subplot(1,3,3)
sns.scatterplot(
    vws_dataset.sample(4000, random_state=42),
    x="blh",
    y="WIND"
)

plt.suptitle("Box plot of Boundary Layer Height and its trend")
plt.tight_layout()
plt.show()

In [ ]:
#ins10

#blh

plt.figure(figsize=(18,6))

plt.subplot(1,3,1)
sns.histplot(
    vws_dataset["i10fg"],
    bins=40
)

plt.subplot(1,3,2)
sns.boxplot(
    vws_dataset,
    x="month",
    y="i10fg"
)

plt.subplot(1,3,3)
sns.scatterplot(
    vws_dataset.sample(4000, random_state=42),
    x="i10fg",
    y="WIND"
)

plt.show()

In [ ]:
#blh

plt.figure(figsize=(18,6))

plt.subplot(1,3,1)
sns.boxplot(
    vws_dataset,
    x="month",
    y="tcc"
)
plt.ylabel("Total Cloud Cover")

plt.subplot(1,3,2)
sns.boxplot(
    vws_dataset,
    x="month",
    y="tp"
)
plt.ylabel("Total Percipitation")

plt.subplot(1,3,3)
sns.boxplot(
    vws_dataset,
    x="month",
    y="ssrd"
)
plt.ylabel("Surface Solar Radition")

plt.suptitle("Trend of different Weather Variables")
plt.tight_layout()
plt.show()

In [ ]:
vws_data = vws_dataset

In [ ]:
# removing timeline variables for correlation matrix
del vws_data["timestamp"]

In [ ]:
#correlation matrix

corr_matrix = vws_data.corr(numeric_only=True)

plt.figure(figsize=(10,6))

sns.heatmap(
    corr_matrix,
    annot=True, #show values
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation Matrix")
plt.show()


In [ ]:
#correlation with target variable

corr_ranking = (
    corr_matrix["WIND"]
    .drop("WIND")
    .sort_values(key=abs, ascending=False)
)

plt.figure(figsize=(10,8))

corr_ranking.plot(
    kind="bar",
    color=["green" if x>0 else "red" for x in corr_ranking]
)

plt.xlabel("Correlation Coefficient")
plt.ylabel("Variables")
plt.title("Correlation Ranking with Wind Generation")

plt.tight_layout()
plt.show()

In [ ]:
multi_corr = corr_matrix.copy()

In [ ]:
multi_corr.columns

In [ ]:
del multi_corr["WIND"]

In [ ]:
multi_corr.head()

In [ ]:
plt.figure(figsize=(10,8))

sns.heatmap(
    multi_corr,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.show()

In [ ]:
%reset